In [1]:
import pandas as pd
import tqdm, os
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../..")

from utils.data_processing import dataset_df_to_token_df
from utils.scaling import fit_scalers, apply_scaler, describe
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aemri75anno_articulatory_features

import torch
import torch.nn as nn
import torch.nn.functional as F

from utils.models import SequenceAutoencoder
from utils.train import (collate_sequence, estimate_loss_scales, train_step,
                   latent_with_duration)

BATCH_SIZE = 32
MIN_FRAMES = 0            # a 1-2 frame token has no trajectory to speak of
ALPHA = 0.5               # static vs velocity, meaningful because scales are fixed
DEVICE = "cpu"

In [2]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_aemri75anno.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])
token_df = dataset_df_to_token_df(df)

In [3]:
token_df

,start,end,phone,art_f,ac_f,subject
0,21.490000,22.380000,ʌ,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub013/2drt/video/su...,sub013
1,6.432471,6.573288,p,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub013/2drt/video/su...,sub013
2,23.649124,24.710000,u,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub013/2drt/video/su...,sub013
3,31.310000,31.502481,i,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub013/2drt/video/su...,sub013
4,19.190000,19.380000,i,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub013/2drt/video/su...,sub013
...,...,...,...,...,...,...
37844,28.780000,28.880000,b,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub072/2drt/video/su...,sub072
37845,27.850000,27.880000,ʌ,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub072/2drt/video/su...,sub072
37846,3.470000,3.500000,f,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub072/2drt/video/su...,sub072
37847,28.220000,28.340000,t,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub072/2drt/video/su...,sub072


In [4]:
if os.path.isfile(f"scalers/aemri75anno_art.pkl"):
    with open(f"scalers/aemri75anno_art.pkl", "rb") as f:
        scalers = pickle.load(f)
else:

    def raw_features(row):

        m = get_aemri75anno_articulatory_features(
                row["art_f"], row["start"], row["end"])
        if m is None or m.shape[0] < MIN_FRAMES:
            return None
        return m.reshape(-1, 174, 2)

    scalers = fit_scalers(token_df, raw_features, mode="isotropic")
    describe(scalers, name="MRI ")

    with open(f"scalers/aemri75anno_art.pkl", "wb") as f:
        pickle.dump(scalers, f)

In [5]:
# ==============================================================================
# Feature extraction, factored out so the scale pass and the train pass agree
# ==============================================================================

def token_features(data, scalers):
    art = get_aemri75anno_articulatory_features(
        data["art_f"], data["start"], data["end"])
    if art is None or art.shape[0] < MIN_FRAMES:
        return None
    art = art.reshape(art.shape[0], 174, 2)
    art = apply_scaler(art, scalers[data["subject"]])
    return torch.tensor(art, dtype=torch.float32), float(data["end"] - data["start"])


def iter_batches(items, batch_size=BATCH_SIZE, progress=True, desc=None, skip=0):
    """Yields (x, lengths, durations), padding to each batch's own max length.

    progress: True for its own bar, False/None for silence, or an existing
              tqdm instance to tick once per batch.
    skip:     leading batches to drop, for resuming mid-epoch.
    """
    n_batches = (len(items) + batch_size - 1) // batch_size

    if progress is True:
        bar, own_bar = tqdm.tqdm(total=n_batches, initial=skip,
                                 desc=desc, unit="batch"), True
    elif progress is False or progress is None:
        bar, own_bar = None, False
    else:
        bar, own_bar = progress, False          # caller-supplied bar

    try:
        for i in range(skip * batch_size, len(items), batch_size):
            yield collate_sequence(items[i:i + batch_size])
            if bar is not None:
                bar.update(1)
    finally:
        if own_bar:
            bar.close()

In [6]:
items = []
for idx, row in tqdm.tqdm(token_df.iterrows(), total=len(token_df)):
    item = token_features(row, scalers)
    if item is None:
        continue
    items.append(item)

100%|██████████| 37849/37849 [1:44:00<00:00,  6.06it/s]


In [7]:
# ==============================================================================
# Pass 0: fix the loss scales once, so ALPHA means something stable
# ==============================================================================

import random
order = list(range(len(items)))
random.Random(42).shuffle(order)
shuffled_items = [items[i] for i in order]

SCALES = estimate_loss_scales(
    ((x, L) for x, L, _ in iter_batches(shuffled_items[:2000])),
    time_dim=1, max_batches=50)
print(f"loss scales: var(x)={SCALES[0]:.5f}  var(dx)={SCALES[1]:.5f}  "
      f"ratio {SCALES[0]/max(SCALES[1],1e-12):.1f}x")

 79%|███████▉  | 50/63 [00:00<00:00, 439.65batch/s]

loss scales: var(x)=0.49114  var(dx)=0.07787  ratio 6.3x


In [8]:
# ==============================================================================
# Config
# ==============================================================================

CKPT_DIR   = "ckpts/aemri75anno"
CKPT_PATH  = f"{CKPT_DIR}/art_model_ckpt.pth"
LOG_PATH   = f"{CKPT_DIR}/art_model_training_log.csv"
SAVE_EVERY = 50
N_EPOCHS   = 3
LOG_KEYS   = ("total", "static", "delta", "duration")

# Set these only to force a resume point (e.g. recovering from an old-style
# checkpoint that stored a bare state_dict). Leave as None normally.
RESUME_EPOCH = None
RESUME_BATCH = None

os.makedirs(CKPT_DIR, exist_ok=True)

# ==============================================================================
# Model
# ==============================================================================

art_model = SequenceAutoencoder(
    feature_shape=(174, 2),
    latent_dim=64,        # was 1024. With ~30k tokens that was memorisation
                          # territory, and Euclidean distances concentrate badly
                          # in 1024-d, which is bad for what you use z for.
).to(DEVICE)

art_optimizer = torch.optim.Adam(art_model.parameters(), lr=1e-3)

log = {"epoch": [], "batch_idx": [], **{k: [] for k in LOG_KEYS}}

# ==============================================================================
# Resume
# ==============================================================================

def save_ckpt(epoch, batch_idx):
    """epoch/batch_idx = the last COMPLETED step. Written atomically so an
    interrupt during saving can't leave a truncated checkpoint."""
    torch.save(
        {
            "epoch": epoch,
            "batch_idx": batch_idx,
            "model": art_model.state_dict(),
            "optimizer": art_optimizer.state_dict(),
        },
        CKPT_PATH + ".tmp",
    )
    os.replace(CKPT_PATH + ".tmp", CKPT_PATH)
    pd.DataFrame(log).to_csv(LOG_PATH, index=False)


start_epoch, start_batch, have_position = 1, 0, False

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)

    if isinstance(ckpt, dict) and "model" in ckpt:
        art_model.load_state_dict(ckpt["model"])
        art_optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch, start_batch = ckpt["epoch"], ckpt["batch_idx"]
        have_position = True
        print(f"Resuming: epoch {start_epoch}, after batch {start_batch}")
    else:
        # Old format: bare state_dict, no optimizer state and no position.
        art_model.load_state_dict(ckpt)
        print("Loaded weights from an old-style checkpoint. "
              "Optimizer state is gone (expect a brief loss bump) and the "
              "position is unknown — set RESUME_EPOCH / RESUME_BATCH if you "
              "know where it stopped.")

    if RESUME_EPOCH is not None:
        start_epoch, start_batch = RESUME_EPOCH, (RESUME_BATCH or 0)
        have_position = True

    # Reload the log, dropping anything recorded after the checkpoint point
    # (those batches will be re-run, so keeping them would double-count).
    if os.path.exists(LOG_PATH):
        old = pd.read_csv(LOG_PATH)
        if "epoch" not in old.columns:
            old.insert(0, "epoch", start_epoch)
        if have_position:
            old = old[
                (old["epoch"] < start_epoch)
                | ((old["epoch"] == start_epoch) & (old["batch_idx"] <= start_batch))
            ]
        log = {k: old[k].tolist() for k in log}

# ==============================================================================
# Train
# ==============================================================================

art_model.train()

for epoch in range(start_epoch, N_EPOCHS + 1):
    # Same seed as the original run -> same shuffle -> skipping by batch index
    # lands on exactly the samples that were still pending.
    order = list(range(len(items)))
    random.Random(42 + epoch).shuffle(order)          # same семantics as random_state=42+epoch
    shuffled_items = [items[i] for i in order]

    skip = start_batch if epoch == start_epoch else 0
    n_batches = (len(shuffled_items) + BATCH_SIZE - 1) // BATCH_SIZE

    pbar = tqdm.tqdm(total=n_batches, initial=skip, unit="batch")
    for batch_idx, (x, lengths, durations) in enumerate(
            iter_batches(shuffled_items, progress=pbar, desc=f"epoch {epoch}", skip=skip),
            start=skip + 1):

        if batch_idx <= skip:
            continue

        x, lengths, durations = x.to(DEVICE), lengths.to(DEVICE), durations.to(DEVICE)

        stats = train_step(art_model, art_optimizer, x, lengths, durations,
                           alpha=ALPHA, scales=SCALES, duration_mode="off")

        log["epoch"].append(epoch)
        log["batch_idx"].append(batch_idx)
        for k in LOG_KEYS:
            log[k].append(stats[k])

        pbar.set_description(
            f"e{epoch} {batch_idx}: ART {stats['total']:.4f} "
            f"(s {stats['static']:.3f} d {stats['delta']:.3f})")

        if batch_idx % SAVE_EVERY == 0:
            save_ckpt(epoch, batch_idx)

    # End of epoch: next run should start clean at the following epoch.
    save_ckpt(epoch + 1, 0)

save_ckpt(N_EPOCHS + 1, 0)

Resuming: epoch 4, after batch 0


In [3]:
if os.path.isfile(f"scalers/aemri75anno_ac.pkl"):
    with open(f"scalers/aemri75anno_ac.pkl", "rb") as f:
        scalers = pickle.load(f)
else:
    def raw_features(row):
        m = get_mfccs(
                row["ac_f"], row["start"], row["end"])
        if m is None or m.shape[0] < MIN_FRAMES:
            return None
        return m

    scalers = fit_scalers(token_df, raw_features, mode="per_feature")
    describe(scalers, name="MRI ")

    with open(f"scalers/aemri75anno_ac.pkl", "wb") as f:
        pickle.dump(scalers, f)

In [4]:
# ==============================================================================
# Feature extraction, factored out so the scale pass and the train pass agree
# ==============================================================================

def token_features(data, scalers):
    ac = get_mfccs(data["ac_f"], data["start"], data["end"])
    if ac is None or ac.shape[0] < MIN_FRAMES:
        return None
    ac = apply_scaler(ac, scalers[data["subject"]])
    return torch.tensor(ac, dtype=torch.float32), float(data["end"] - data["start"])

def iter_batches(df, scalers, batch_size=BATCH_SIZE, progress=None):
    """Yields (x, lengths, durations), padding to each batch's own max length."""
    buf = []
    rows = df.iterrows() if progress is None else progress
    for _, data in rows:
        item = token_features(data, scalers)
        if item is None:
            continue
        buf.append(item)
        if len(buf) == batch_size:
            yield collate_sequence(buf)
            buf = []
    if buf:                                   # the tail batch, previously dropped
        yield collate_sequence(buf)

In [5]:
N_EPOCHS   = 5

In [6]:
# ==============================================================================
# Pass 0: fix the loss scales once, so ALPHA means something stable
# ==============================================================================

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

SCALES = estimate_loss_scales(
    ((x, L) for x, L, _ in iter_batches(shuffled_token_df.head(2000), scalers)),
    time_dim=1, max_batches=50)
print(f"loss scales: var(x)={SCALES[0]:.5f}  var(dx)={SCALES[1]:.5f}  "
      f"ratio {SCALES[0]/max(SCALES[1],1e-12):.1f}x")

loss scales: var(x)=1.00646  var(dx)=0.29390  ratio 3.4x


In [7]:
# ==============================================================================
# Model
# ==============================================================================

ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,        # was 1024. With ~30k tokens that was memorisation
                          # territory, and Euclidean distances concentrate badly
                          # in 1024-d, which is bad for what you use z for.
).to(DEVICE)

ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=1e-3)

log = {"batch_idx": [], "total": [], "static": [], "delta": [], "duration": []}

# ==============================================================================
# Train
# ==============================================================================

ac_model.train()

for epoch in range(1, N_EPOCHS + 1):
    shuffled_token_df = token_df.sample(frac=1, random_state=42+epoch).reset_index(drop=True)
    pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

    for batch_idx, (x, lengths, durations) in enumerate(
            iter_batches(shuffled_token_df, scalers, progress=pbar), start=1):

        x, lengths, durations = x.to(DEVICE), lengths.to(DEVICE), durations.to(DEVICE)

        # time_dim=1 is inferred from the model class; masking, gradient clipping and
        # the detached duration probe all live in here.
        stats = train_step(ac_model, ac_optimizer, x, lengths, durations,
                        alpha=ALPHA, scales=SCALES, duration_mode="off")

        log["batch_idx"].append(batch_idx)
        for k in ("total", "static", "delta", "duration"):
            log[k].append(stats[k])
        pbar.set_description(
            f"{batch_idx}: AC {stats['total']:.4f} "
            f"(s {stats['static']:.3f} d {stats['delta']:.3f})")

        if batch_idx % 300 == 0:
            torch.save(ac_model.state_dict(), "ckpts/aemri75anno/ac_model_ckpt.pth")
            pd.DataFrame(log).to_csv("ckpts/aemri75anno/ac_model_training_log.csv", index=False)

torch.save(ac_model.state_dict(), "ckpts/aemri75anno/ac_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aemri75anno/ac_model_training_log.csv", index=False)

1133: AC 0.6140 (s 0.387 d 0.841): 100%|██████████| 37849/37849 [05:25<00:00, 116.29it/s]
